# NB5 · Safety guardrails and governance

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---

This notebook does not improve the performance of the model. It determines when the
model should not speak.

There is a single check cell. Verification has passed entirely to you at this stage;
the notebook only makes the results visible.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for module in ['checks.py', 'evaluate.py', 'explain.py', 'safety.py',
               'mimic_web.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{module}', module)

import numpy as np
import pandas as pd
import checks, evaluate as ev, explain as ex, safety as sf

checks.LANG = ev.LANG = sf.LANG = 'en'


In [ ]:
# Fixed cell. Rebuilds the output of the previous notebooks.
import pipeline as pl

state = pl.prepare(verbose=False)
model = state['model']
test = state['test']
features = state['features']
probability = state['probabilities']
y_test = state['y_test']
threshold = ev.threshold_for_sensitivity(y_test, probability, target=0.80)
print(f'Model and predictions ready. Operating threshold: {threshold:.3f}')


---

## Fixed section · The abstention band

The band is not chosen, it is derived from the data. An abstention budget is set and the
band is widened until that budget is spent.

The audit is what matters. The output below compares accuracy inside the band against
accuracy outside it. Where the two are close, the band is not isolating genuine
uncertainty and should be abandoned rather than kept for appearances.


In [ ]:
# Fixed cell. Builds the guardrails and assembles the system.
band = sf.choose_band(y_test, probability, threshold, max_abstain=0.20)
policy = sf.AbstentionPolicy(threshold, band['band'])
drift = sf.DriftDetector().fit(state['train'][features])
validator = sf.PhysiologicalValidator().fit(state['train'][features])
guarded = sf.GuardedModel(model, policy, drift, validator)

for key, value in band.items():
    print(f'{key:<24} {value}')


---

## Step 1 · A guard for missing data

The cell below demonstrates a problem. Observe what the system does for a patient whose
values are entirely missing.


In [ ]:
empty_patient = test[features].iloc[[0]].copy()
empty_patient.loc[empty_patient.index[0], :] = np.nan
print('Decision without a guard:', guarded.predict_one(empty_patient)['decision'])


The imputer fills every gap with the training medians and the model returns a confident
probability. The system proposes a decision for a patient about whom nothing is known.
No error is raised and the code runs flawlessly.

An imputer is added when it is asked for; when it should stand down is not discussed
unless the question is put. The next prompt builds that guard. Where the threshold sits
is a clinical decision: how many of a patient's features may be missing before no
prediction is produced?


### Prompt 1

```
There is an object named guarded with a method called predict_one. It takes a single
row DataFrame and returns a dictionary.

Write a single Python function that wraps it. The function should compute the
proportion of features missing in the incoming row. Where that proportion exceeds a
limit I set, it must produce no prediction and escalate to the clinician on grounds of
insufficient data. Otherwise it returns the result of guarded.predict_one.

Define the limit outside the function as a constant named in capitals, and add a
comment stating that it is a clinical decision rather than a technical default.

CONTRACT
Produce a function named predict_guarded taking a single row DataFrame.
The dictionary it returns must contain the keys decision and probability.
Given a row where every value is missing, probability must be None.
```


In [ ]:
# Paste the generated code into this cell and run it.


### Check 1


In [ ]:
result = predict_guarded(empty_patient)
print('Decision with a guard:', result['decision'])
print('Probability          :', result['probability'])
print()
ordinary = predict_guarded(test[features].iloc[[1]])
print('Ordinary patient     :', ordinary['decision'], '·', ordinary['probability'])


---

## Fixed section · Red team

The cell below constructs five inputs that push the system and shows what it does with
each. One of them is not obviously broken but clinically plausible. That row is the
important one; a system that fails only on obviously broken input has not really been
tested.


In [ ]:
sf.red_team(guarded, test[features])


---

## Governance artefacts

The final step produces two documents: A model card and a regulatory triage note. The
cell below gathers the figures measured across these notebooks into the context block
that goes at the head of the prompt.


In [ ]:
report = ev.honest_report(y_test, probability,
                         groups=test['gender'] if 'gender' in test.columns else None,
                         target_sensitivity=0.80, label='guarded system')

context = f'''SYSTEM SUMMARY
Purpose: Predicting a stay longer than three days, six hours after ICU admission.
Data: MIMIC-IV demo, single centre, {len(state['cohort'])} stays.
Model: Logistic regression, class weighted, split at patient level.

EVALUATION
AUC {report['discrimination']['auc']:.3f} (95% CI {report['discrimination']['ci_low']:.3f}-{report['discrimination']['ci_high']:.3f})
Calibration slope {report['calibration']['slope']:.2f}
Threshold {report['threshold']:.3f}: sensitivity {report['operating_point']['sensitivity']:.3f}, PPV {report['operating_point']['ppv']:.3f}
Per 100 patients {report['clinical']['alerts_fired']} alerts, {report['clinical']['true_alerts']} true.

GUARDRAILS
Abstention band {band['low']:.3f}-{band['high']:.3f}, covering {band['abstain_fraction']:.0%} of cases.
Band audit: {band['verdict']}
A missing data guard is in place.
'''
print(context)


### Prompt 2

Copy the context block above and place it at the head of the prompt below. This prompt
produces documents rather than code.

```
Produce two documents for this system. Return markdown, not code.

DOCUMENT 1: A model card. Intended use and users; out-of-scope uses; training data
and its limitations; evaluation results including the subgroup breakdown; the
operating threshold and who chose it; known failure modes; abstention and escalation
behaviour; the post-deployment monitoring plan.

DOCUMENT 2: A regulatory triage note answering these four questions:
- Under EU Medical Device Regulation 2017/745 Rule 11, would this software qualify as
  a medical device, and on what reasoning?
- If it is a device, which EU AI Act annex applies and what is the operative
  compliance date? Note that the Digital Omnibus, Regulation (EU) 2026/1744, which
  entered into force in July 2026, changed these dates.
- Does it meet the four criteria of the clinical decision support exclusion under
  FD&C Act 520(o)(1)(E)? Address the independent review criterion specifically.
- Which obligations arise under data protection law given that health data is a
  special category of personal data?

For each of the four points, state your confidence and what a lawyer would need to
check. Present none of it as legal advice.
```


## Auditing the documents produced

Read the regulatory triage note carefully. The compliance dates in this area changed in
July 2026, and the change is recent enough that many tools still return the superseded
timetable. If the tool gives the old date, that is a concrete instance of confident
invention and a behaviour worth keeping in mind for the rest of the workshop.

Request a DOI or official document number for every citation in the documents. Where
none can be given, the citation should be removed.

Compare the model card produced against `templates/en/model-card.md` in the repository.
Every heading present in the template and absent from the document is an unanswered
question.
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The
MIMIC-IV demo data comes from a single hospital in the United States and does not
represent an intensive care population elsewhere. The material is for teaching.
